# 01 — Per-stage walkthrough: merge → radius → fit-setup → bin

`midas-transforms` is the pure-Python / PyTorch port of the four
C binaries that sit between peak-fitting and indexing in the FF-HEDM
workflow:

| Stage | C binary | Python entry |
|---|---|---|
| 1. merge overlapping peaks | `MergeOverlappingPeaksAllZarr` | `merge_overlapping_peaks` |
| 2. compute ring radii      | `CalcRadiusAllZarr`           | `calc_radius` |
| 3. transform + filter      | `FitSetupZarr`                | `fit_setup` |
| 4. bin for the indexer     | `SaveBinData`                 | `bin_data` |

This notebook runs all four **stage-by-stage on CPU**, feeding each
stage's in-memory output to the next (every stage takes a NumPy
array and a `write=False` flag). The input is a small **synthetic**
FF dataset (3 rings × 4 η × 4 ω frames) — no zarr file needed at
this layer.


In [1]:
import math
import numpy as np
from midas_transforms.merge.core import (
    COL_SPOTID, COL_II, COL_OMEGA, COL_YCEN, COL_ZCEN, COL_IMAX,
    COL_RADIUS, COL_ETA, COL_SIGMAR, COL_SIGMAETA, COL_NRPX,
    COL_NRPXTOT, COL_RAWSUM, N_PEAK_COLS,
)

# Geometry shared across the notebook.
NR_PIXELS = 2048
PX_UM = 200.0          # pixel size
LSD_UM = 1_000_000.0   # sample-detector distance
YCEN = ZCEN = 1024.0   # beam center (px)
RINGS = [1, 2, 3]
RING_RADII_UM = [94949.4, 109761.7, 155930.4]   # ring radii in µm
N_FRAMES = 4
ETAS = [-120.0, -40.0, 40.0, 120.0]

def synth_frames():
    """One (n_peaks, 29) AllPeaks_PS-layout array per ω frame.

    Each peak is placed on its ring at the chosen η, so its observed
    radius matches the hkls ring radius and survives calc_radius.
    """
    frames = []
    sid = 1
    for fi in range(N_FRAMES):
        rows = []
        for rr in RING_RADII_UM:
            r_px = rr / PX_UM
            for eta in ETAS:
                y = YCEN - r_px * math.sin(math.radians(eta))
                z = ZCEN + r_px * math.cos(math.radians(eta))
                row = np.zeros(N_PEAK_COLS)
                row[COL_SPOTID] = sid; sid += 1
                row[COL_II] = 100.0
                row[COL_OMEGA] = fi * 0.25
                row[COL_YCEN] = y; row[COL_ZCEN] = z
                row[COL_IMAX] = 200.0
                row[COL_RADIUS] = r_px        # radial distance (px) → ring radius
                row[COL_ETA] = eta
                row[COL_SIGMAR] = 1.0; row[COL_SIGMAETA] = 1.0
                row[COL_NRPX] = 9; row[COL_NRPXTOT] = 9
                row[COL_RAWSUM] = 100.0
                rows.append(row)
        frames.append(np.array(rows))
    return frames

def write_hkls(path):
    with open(path, "w") as f:
        f.write("h k l D-spacing RingNr g1 g2 g3 Theta 2Theta Radius\n")
        for rn, rr in zip(RINGS, RING_RADII_UM):
            f.write(f"1 1 1 2.0 {rn} 0 0 0 1.0 2.0 {rr}\n")

import tempfile
from pathlib import Path
import torch
from midas_transforms.params import ZarrParams

print("torch:", torch.__version__, "| device: cpu")
WORK = Path(tempfile.mkdtemp(prefix="midas_transforms_nb_"))
write_hkls(WORK / "hkls.csv")
print("workspace:", WORK)


torch: 2.11.0 | device: cpu
workspace: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_transforms_nb_ty5cp6wx


## 0. Parameters

The C stages read geometry from the Zarr's `analysis_parameters`
group; here we build the equivalent `ZarrParams` object directly. The
ring radii and `Width` (radial match window, µm) gate which peaks
survive `calc_radius`.


In [2]:
zp = ZarrParams()
zp.Lsd = LSD_UM
zp.Wavelength = 0.18
zp.PixelSize = PX_UM
zp.YCen = YCEN
zp.ZCen = ZCEN
zp.NrPixels = NR_PIXELS
zp.SpaceGroup = 225
zp.LatticeConstant = (3.6, 3.6, 3.6, 90.0, 90.0, 90.0)
zp.RingThresh = [(rn, 100.0) for rn in RINGS]
zp.OmegaStart = 0.0
zp.OmegaStep = 0.25
zp.Width = 2000.0          # radial match window (µm)
zp.Hbeam = 2000.0
zp.Rsample = 1000.0
zp.MarginRadius = zp.MarginRadial = 2000.0
zp.MarginEta = 500.0
zp.MarginOme = 2.0
zp.EtaBinSize = zp.OmeBinSize = 5.0
zp.StepSizeOrient = 0.2
zp.StepSizePos = 100.0
zp.EndNr = N_FRAMES
print("rings:", RINGS, "| radii (µm):", RING_RADII_UM)


rings: [1, 2, 3] | radii (µm): [94949.4, 109761.7, 155930.4]


## Stage 1 — merge overlapping peaks

`merge_overlapping_peaks` consolidates peaks that recur across
adjacent ω frames into single spots. We pass the synthetic frames
in-memory via `frames=` (bypassing the on-disk `AllPeaks_PS.bin`)
and `write=False`. Output is the `(N, 17)` merged spot table.


In [3]:
from midas_transforms import merge_overlapping_peaks

frames = synth_frames()
n_in = sum(f.shape[0] for f in frames)
merged = merge_overlapping_peaks(
    frames=frames, result_folder=WORK,
    nr_pixels=NR_PIXELS, device="cpu", dtype="float64", write=False,
)
merge_arr = merged.peaks.cpu().numpy()
print(f"in: {n_in} raw peaks across {len(frames)} frames "
      f"→ out: {merge_arr.shape[0]} merged spots, {merge_arr.shape[1]} cols")


in: 48 raw peaks across 4 frames → out: 12 merged spots, 17 cols


## Stage 2 — compute ring radii

`calc_radius` matches each merged spot to its Bragg ring (radial
window `|R_obs - R_ring| < Width`), computes 2θ, and emits the
`(N, 24)` radius table. The ring radii come from `hkls.csv`.


In [4]:
from midas_transforms import calc_radius

radius = calc_radius(
    result_folder=WORK, zarr_params=zp, result_array=merge_arr,
    start_nr=1, end_nr=N_FRAMES, device="cpu", dtype="float64", write=False,
)
radius_arr = radius.spots.cpu().numpy()
print("radius table:", radius_arr.shape, "(N, 24)")
print("ring numbers found:", sorted(set(int(r) for r in radius_arr[:, 13])))


radius table: (12, 24) (N, 24)
ring numbers found: [1, 2, 3]


## Stage 3 — transform + filter (fit-setup)

`fit_setup` applies the detector transform (tilts / distortion / ω
correction), filters to in-spec spots, and produces the 8-column
`InputAll` + the extra-info matrix that `bin_data` consumes. With
`DoFit==1` it would also refine geometry; here we keep it as the
pass-through transform.


In [5]:
from midas_transforms import fit_setup

fs = fit_setup(
    result_folder=WORK, zarr_params=zp, radius_array=radius_arr,
    start_nr=1, end_nr=N_FRAMES, device="cpu", dtype="float64", write=False,
)
inputall = fs.spots_inputall.cpu().numpy()
extra = fs.extra.cpu().numpy()
print("InputAll:", inputall.shape, "(N, 8)")
print("ExtraInfo:", extra.shape)
print("InputAll cols = [YLab, ZLab, Omega, GrainRadius, SpotID, RingNr, Eta, Ttheta]")
print("first spot:", np.round(inputall[0], 3))


InputAll: (12, 8) (N, 8)
ExtraInfo: (12, 18)
InputAll cols = [YLab, ZLab, Omega, GrainRadius, SpotID, RingNr, Eta, Ttheta]
first spot: [-8.2228592e+04 -4.7474700e+04  3.7500000e-01  1.4145600e+02
  1.0000000e+00  1.0000000e+00  1.2000000e+02  5.4240000e+00]


## Stage 4 — bin for the indexer

`bin_data` is the drop-in for `SaveBinData`. It writes the four
binaries the indexer reads: `Spots.bin`, `ExtraInfo.bin`, and the
`Data.bin` / `nData.bin` spatial-bin pair. We pass
the fit-setup arrays in-memory and `write=True` to land the files.


In [6]:
from midas_transforms import bin_data

bins = bin_data(
    result_folder=WORK, paramstest=fs.paramstest,
    spots_inputall=inputall, extra_inputall=extra,
    device="cpu", dtype="float64", write=True,
)
print("Spots.bin   :", tuple(bins.spots.shape))
print("ExtraInfo   :", tuple(bins.extra_info.shape))
print("Data / nData:", tuple(bins.data.shape), "/", tuple(bins.ndata.shape))
print("eta/ome/ring bins:", bins.n_eta_bins, bins.n_ome_bins, bins.n_ring_bins)
print()
print("files written to workspace:")
for p in sorted(WORK.glob("*.bin")):
    print(f"  {p.name:16s} {p.stat().st_size} bytes")


Spots.bin   : (12, 10)
ExtraInfo   : (12, 16)
Data / nData: (0,) / (15552, 2)
eta/ome/ring bins: 72 72 3

files written to workspace:
  Data.bin         0 bytes
  ExtraInfo.bin    1536 bytes
  Spots.bin        960 bytes
  Spots_det.bin    48 bytes
  nData.bin        248832 bytes


## Recap

We ran the full FF intermediate chain stage-by-stage, threading each
stage's in-memory array into the next:

```
frames → merge_overlapping_peaks → calc_radius → fit_setup → bin_data
```

The `*.bin` files now in the workspace are exactly what
[`midas-index`](../../midas_index/) consumes. Notebook
[02](02_pipeline_from_zarr.ipynb) runs the same four stages as one
chained `Pipeline.from_zarr(...)` call, reading a synthetic Zarr
archive end-to-end.
